In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load processed data
df = pd.read_csv('../data/processed/hotel_bookings_processed.csv')

print(f"Dataset shape: {df.shape}")
print(f"Price range: ${df['adr'].min():.2f} - ${df['adr'].max():.2f}")

# =============================================================================
# 1. DEMAND CURVE ANALYSIS
# =============================================================================

def analyze_demand_curve(df, price_col='adr', segments=None):
    """
    Analyze how demand varies with price across different segments
    """
    results = {}
    
    if segments is None:
        segments = {'All': df}
    else:
        segments = {seg: df[df[segments] == seg] for seg in df[segments].unique()}
    
    for segment_name, segment_df in segments.items():
        # Create price bins
        segment_df = segment_df.copy()
        segment_df['price_bin'] = pd.cut(segment_df[price_col], bins=15, duplicates='drop')
        
        # Calculate demand metrics for each price bin
        demand_analysis = segment_df.groupby('price_bin').agg({
            price_col: ['mean', 'count'],  # Average price and booking count
            'total_nights': 'mean',
            'total_guests': 'mean',
            'lead_time': 'mean'
        }).round(2)
        
        demand_analysis.columns = ['avg_price', 'booking_count', 'avg_nights', 'avg_guests', 'avg_lead_time']
        demand_analysis = demand_analysis.reset_index()
        
        # Calculate revenue per price bin
        demand_analysis['total_revenue'] = demand_analysis['avg_price'] * demand_analysis['booking_count']
        demand_analysis['revenue_per_guest'] = demand_analysis['total_revenue'] / demand_analysis['avg_guests']
        
        results[segment_name] = demand_analysis
    
    return results

# Analyze overall demand curve
demand_curves = analyze_demand_curve(df)
print("=== DEMAND CURVE ANALYSIS ===")
print(demand_curves['All'])

# =============================================================================
# 2. PRICE ELASTICITY CALCULATION
# =============================================================================

def calculate_price_elasticity(df, price_col='adr', demand_proxy='total_nights'):
    """
    Calculate price elasticity of demand using multiple methods
    """
    # Method 1: Simple correlation-based elasticity
    price_log = np.log(df[price_col] + 1)
    demand_log = np.log(df[demand_proxy] + 1)
    
    correlation_elasticity = np.corrcoef(price_log, demand_log)[0,1]
    
    # Method 2: Regression-based elasticity
    X = price_log.values.reshape(-1, 1)
    y = demand_log.values
    
    reg_model = LinearRegression()
    reg_model.fit(X, y)
    regression_elasticity = reg_model.coef_[0]
    
    # Method 3: Segment-wise elasticity
    segments = ['hotel', 'market_segment', 'customer_type']
    segment_elasticities = {}
    
    for segment in segments:
        if segment in df.columns:
            seg_elasticities = []
            for seg_value in df[segment].unique():
                seg_data = df[df[segment] == seg_value]
                if len(seg_data) > 50:  # Minimum data points
                    seg_price_log = np.log(seg_data[price_col] + 1)
                    seg_demand_log = np.log(seg_data[demand_proxy] + 1)
                    if seg_price_log.std() > 0 and seg_demand_log.std() > 0:
                        seg_elasticity = np.corrcoef(seg_price_log, seg_demand_log)[0,1]
                        seg_elasticities.append(seg_elasticity)
            
            segment_elasticities[segment] = {
                'mean': np.mean(seg_elasticities),
                'std': np.std(seg_elasticities),
                'values': seg_elasticities
            }
    
    return {
        'correlation_elasticity': correlation_elasticity,
        'regression_elasticity': regression_elasticity,
        'segment_elasticities': segment_elasticities
    }

# Calculate elasticity
elasticity_results = calculate_price_elasticity(df)

print("\n=== PRICE ELASTICITY ANALYSIS ===")
print(f"Overall Correlation Elasticity: {elasticity_results['correlation_elasticity']:.3f}")
print(f"Regression-based Elasticity: {elasticity_results['regression_elasticity']:.3f}")

for segment, values in elasticity_results['segment_elasticities'].items():
    print(f"{segment} Average Elasticity: {values['mean']:.3f} (±{values['std']:.3f})")

# =============================================================================
# 3. ADVANCED REVENUE OPTIMIZATION
# =============================================================================

class RevenueOptimizer:
    def __init__(self, demand_model, base_elasticity=-0.5):
        self.demand_model = demand_model
        self.base_elasticity = base_elasticity
        self.feature_cols = [
            'lead_time', 'total_nights', 'total_guests', 'is_weekend', 'is_peak_season',
            'day_of_week', 'arrival_date_month_num', 'is_repeated_guest', 'previous_cancellations',
            'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests',
            'hotel_encoded', 'meal_encoded', 'market_segment_encoded', 'distribution_channel_encoded',
            'reserved_room_type_encoded', 'deposit_type_encoded', 'customer_type_encoded'
        ]
    
    def predict_demand(self, features, price):
        """Predict demand at a given price using elasticity adjustment"""
        # Get base demand prediction
        base_demand = self.demand_model.predict(features)[0]
        
        # Apply price elasticity
        # Demand = Base_Demand * (Price / Base_Price)^elasticity
        base_price = base_demand  # Using predicted price as base
        if base_price > 0:
            price_ratio = price / base_price
            adjusted_demand = base_demand * (price_ratio ** self.base_elasticity)
        else:
            adjusted_demand = base_demand
        
        return max(0, adjusted_demand)  # Demand cannot be negative
    
    def optimize_price(self, features, price_range=(50, 500), steps=100, 
                      cost_per_night=30, min_margin=0.2):
        """
        Find optimal price considering revenue, costs, and constraints
        """
        prices = np.linspace(price_range[0], price_range[1], steps)
        results = []
        
        for price in prices:
            # Predict demand at this price
            demand = self.predict_demand(features, price)
            
            # Calculate metrics
            revenue = price * demand
            cost = cost_per_night * demand
            profit = revenue - cost
            margin = (profit / revenue) if revenue > 0 else 0
            
            # Check constraints
            meets_margin = margin >= min_margin
            
            results.append({
                'price': price,
                'demand': demand,
                'revenue': revenue,
                'profit': profit,
                'margin': margin,
                'meets_constraints': meets_margin
            })
        
        results_df = pd.DataFrame(results)
        
        # Find optimal prices for different objectives
        valid_results = results_df[results_df['meets_constraints']]
        
        if len(valid_results) > 0:
            optimal_revenue = valid_results.loc[valid_results['revenue'].idxmax()]
            optimal_profit = valid_results.loc[valid_results['profit'].idxmax()]
        else:
            optimal_revenue = results_df.loc[results_df['revenue'].idxmax()]
            optimal_profit = results_df.loc[results_df['profit'].idxmax()]
        
        return {
            'all_results': results_df,
            'optimal_revenue': optimal_revenue,
            'optimal_profit': optimal_profit
        }
    
    def scenario_analysis(self, features, scenarios):
        """
        Analyze multiple pricing scenarios
        """
        scenario_results = {}
        
        for scenario_name, scenario_params in scenarios.items():
            results = self.optimize_price(features, **scenario_params)
            scenario_results[scenario_name] = results
        
        return scenario_results

# Load the trained model
import joblib
model = joblib.load('../src/models/demand_model.pkl')

# Create optimizer
optimizer = RevenueOptimizer(model)

# Example optimization for a sample booking
sample_features = df[optimizer.feature_cols].iloc[[100]]  # Get one sample
optimization_results = optimizer.optimize_price(sample_features)

print("\n=== REVENUE OPTIMIZATION RESULTS ===")
print(f"Optimal for Revenue: ${optimization_results['optimal_revenue']['price']:.2f}")
print(f"Expected Revenue: ${optimization_results['optimal_revenue']['revenue']:.2f}")
print(f"Expected Demand: {optimization_results['optimal_revenue']['demand']:.2f}")
print(f"Profit Margin: {optimization_results['optimal_revenue']['margin']:.1%}")

print(f"\nOptimal for Profit: ${optimization_results['optimal_profit']['price']:.2f}")
print(f"Expected Profit: ${optimization_results['optimal_profit']['profit']:.2f}")
print(f"Expected Revenue: ${optimization_results['optimal_profit']['revenue']:.2f}")
print(f"Profit Margin: {optimization_results['optimal_profit']['margin']:.1%}")

# =============================================================================
# 4. VISUALIZATIONS
# =============================================================================

# Create comprehensive visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Demand vs Price', 'Revenue Optimization', 
                   'Price Elasticity by Segment', 'Seasonal Price Patterns'],
    specs=[[{"secondary_y": False}, {"secondary_y": True}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Plot 1: Demand Curve
demand_data = demand_curves['All']
fig.add_trace(
    go.Scatter(x=demand_data['avg_price'], y=demand_data['booking_count'],
              mode='markers+lines', name='Bookings vs Price'),
    row=1, col=1
)

# Plot 2: Revenue Optimization
opt_data = optimization_results['all_results']
fig.add_trace(
    go.Scatter(x=opt_data['price'], y=opt_data['revenue'],
              mode='lines', name='Revenue', line=dict(color='blue')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=opt_data['price'], y=opt_data['profit'],
              mode='lines', name='Profit', line=dict(color='green')),
    row=1, col=2, secondary_y=True
)

# Plot 3: Seasonal patterns
seasonal_data = df.groupby('arrival_date_month_num').agg({
    'adr': 'mean',
    'total_nights': 'sum'
}).reset_index()

fig.add_trace(
    go.Bar(x=seasonal_data['arrival_date_month_num'], 
           y=seasonal_data['adr'], name='Average Price'),
    row=2, col=2
)

# Update layout
fig.update_layout(height=800, showlegend=True, 
                 title_text="Dynamic Pricing Analysis Dashboard")
fig.show()

# Save optimization results
optimization_results['all_results'].to_csv('../data/processed/optimization_results.csv', index=False)
print("\n✅ Optimization results saved!")


Dataset shape: (73419, 47)
Price range: $0.26 - $510.00
=== DEMAND CURVE ANALYSIS ===
             price_bin  avg_price  booking_count  avg_nights  avg_guests  \
0      (-0.25, 34.243]      26.36           1662        3.47        1.54   
1     (34.243, 68.225]      54.31          16096        3.47        1.68   
2    (68.225, 102.208]      85.40          24781        3.39        1.85   
3   (102.208, 136.191]     117.70          16677        3.35        1.99   
4   (136.191, 170.173]     151.66           8007        3.42        2.25   
5   (170.173, 204.156]     185.53           3397        3.63        2.58   
6   (204.156, 238.139]     219.34           1660        3.97        2.77   
7   (238.139, 272.121]     252.55            729        3.80        2.96   
8   (272.121, 306.104]     287.87            255        3.80        3.15   
9   (306.104, 340.087]     321.59            111        4.17        3.44   
10  (340.087, 374.069]     355.26             30        4.20        3.57   
11


✅ Optimization results saved!
